In [1]:
# Cell 1 — Import
import sys
import json
import os
sys.path.append('../')
from dotenv import load_dotenv
from neo4j import GraphDatabase

load_dotenv('../.env')

driver = GraphDatabase.driver(
    os.getenv("NEO4J_URI"),
    auth=(os.getenv("NEO4J_USERNAME"),
          os.getenv("NEO4J_PASSWORD"))
)
print("Kết nối Neo4j thành công")

Kết nối Neo4j thành công


In [2]:
# Cell 2 — Đọc regulation JSON
with open('../data/raw/sop/regulation_graph.json', 'r') as f:
    reg = json.load(f)

print(f"Document: {reg['document']['title']}")
print(f"Activities: {len(reg['activities'])}")
print(f"Sequences:  {len(reg['sequences'])}")
print(f"Conditions: {len(reg['conditions'])}")
print(f"Roles:      {len(reg['roles'])}")

Document: Directive 2008/48/EC on Credit Agreements for Consumers
Activities: 13
Sequences:  6
Conditions: 4
Roles:      3


In [6]:
# Cell 3 — Tạo Document node
with driver.session() as session:
    d = reg['document']
    session.run("""
        MERGE (doc:Document {id: $id})
        SET doc.title          = $title,
            doc.issuer         = $issuer,
            doc.effective_date = $effective_date,
            doc.jurisdiction   = $jurisdiction
    """, **d)
print("Tạo Document node xong")

Tạo Document node xong


In [7]:
# Cell 4 — Tạo Activity nodes
with driver.session() as session:
    for act in reg['activities']:
        session.run("""
            MERGE (a:Activity {id: $id})
            SET a.name         = $name,
                a.description  = $description,
                a.event_origin = $event_origin,
                a.article_ref  = $article_ref,
                a.required     = $required
            WITH a
            MATCH (doc:Document {id: $doc_id})
            MERGE (a)-[:DEFINED_IN]->(doc)
        """, doc_id=reg['document']['id'], **act)
print(f"Tạo {len(reg['activities'])} Activity node xong")

Tạo 13 Activity node xong


In [8]:
# Cell 5 — Tạo MUST_PRECEDE edges
with driver.session() as session:
    for seq in reg['sequences']:
        session.run("""
            MATCH (a1:Activity {id: $from_id})
            MATCH (a2:Activity {id: $to_id})
            MERGE (a1)-[r:MUST_PRECEDE]->(a2)
            SET r.article_ref  = $article_ref,
                r.description  = $description,
                r.sequence_id  = $id
        """, from_id=seq['from'],
             to_id=seq['to'],
             **{k: v for k, v in seq.items()
                if k not in ['from', 'to']})
print(f"Tạo {len(reg['sequences'])} MUST_PRECEDE edge xong")

Tạo 7 MUST_PRECEDE edge xong


In [10]:
# Cell 6 — Tạo Condition nodes
with driver.session() as session:
    for cond in reg['conditions']:
        session.run("""
            MERGE (c:Condition {id: $id})
            SET c.description    = $description,
                c.expression     = $expression,
                c.article_ref    = $article_ref,
                c.violation_type = $violation_type
            WITH c
            MATCH (doc:Document {id: $doc_id})
            MERGE (c)-[:DEFINED_IN]->(doc)
        """, doc_id=reg['document']['id'],
             **{k: v for k, v in cond.items()
                if k not in ['threshold', 'unit']})
print(f"Tạo {len(reg['conditions'])} Condition node xong")

Tạo 4 Condition node xong


In [12]:
# Cell 7 — Tạo Role nodes và PERFORMED_BY edges
with driver.session() as session:
    for role in reg['roles']:
        # Tạo Role node
        session.run("""
            MERGE (r:Role {id: $id})
            SET r.name = $name,
                r.type = $type
        """, id=role['id'],
             name=role['name'],
             type=role['type'])

        # Tạo PERFORMED_BY từ Activity → Role
        for act_id in role['performs']:
            session.run("""
                MATCH (a:Activity {id: $act_id})
                MATCH (r:Role {id: $role_id})
                MERGE (a)-[:PERFORMED_BY]->(r)
            """, act_id=act_id, role_id=role['id'])

print(f"Tạo {len(reg['roles'])} Role node xong")

Tạo 3 Role node xong


In [13]:
# Cell 8 — Kiểm tra Regulation Graph
with driver.session() as session:
    r1 = session.run(
        "MATCH (n:Activity) RETURN count(n) AS cnt"
    )
    r2 = session.run(
        "MATCH ()-[r:MUST_PRECEDE]->() RETURN count(r) AS cnt"
    )
    r3 = session.run(
        "MATCH (n:Condition) RETURN count(n) AS cnt"
    )
    r4 = session.run(
        "MATCH (n:Role) RETURN count(n) AS cnt"
    )
    print("Regulation Graph trong Neo4j:")
    print(f"  Activity node   : {r1.single()['cnt']}")
    print(f"  MUST_PRECEDE    : {r2.single()['cnt']}")
    print(f"  Condition node  : {r3.single()['cnt']}")
    print(f"  Role node       : {r4.single()['cnt']}")

Regulation Graph trong Neo4j:
  Activity node   : 13
  MUST_PRECEDE    : 7
  Condition node  : 4
  Role node       : 3


In [14]:
# Cell 9 — Xem graph trong Neo4j Browser
print("Chạy query sau trong Neo4j Browser:")
print("""
MATCH (a1:Activity)-[r:MUST_PRECEDE]->(a2:Activity)
RETURN a1, r, a2
""")

Chạy query sau trong Neo4j Browser:

MATCH (a1:Activity)-[r:MUST_PRECEDE]->(a2:Activity)
RETURN a1, r, a2



In [6]:
import sys
sys.path.append('../')  # nếu trong thư mục notebooks/
# hoặc sys.path.append('.') nếu ở thư mục gốc

from src.knowledge_graph.regulation_loader import RegulationLoader

# Khởi tạo loader
loader = RegulationLoader(env_path='../.env')  
# hoặc env_path='.env' nếu ở thư mục gốc

# Xoá regulation graph cũ (giữ nguyên Trace Graph)
loader.clear_regulation()

# Load lại từ JSON đã patch
loader.load_from_json('../data/raw/sop/regulation_graph.json')
# hoặc 'data/raw/sop/regulation_graph.json' nếu ở thư mục gốc

# Verify
counts = loader.verify()

# Đóng kết nối
loader.close()

Đã xoá Regulation graph cũ
Đọc file: ../data/raw/sop/regulation_graph.json
  Activities : 13
  Sequences  : 6
  Conditions : 4
  Roles      : 3
  Document: Directive 2008/48/EC on Credit Agreements for Consumers
  13 Activity nodes
  6 MUST_PRECEDE edges
  4 Condition nodes
  5 REQUIRES edges
  3 Role nodes
Load Regulation graph hoàn thành
Regulation loaded: 13 activities, 6 sequences, 4 conditions, 0 requires, 3 roles


In [4]:
# Xem Activity IDs thực tế trong Neo4j
from neo4j import GraphDatabase
from dotenv import load_dotenv
import os

load_dotenv('../.env')
driver = GraphDatabase.driver(
    os.getenv("NEO4J_URI"),
    auth=(os.getenv("NEO4J_USERNAME"),
          os.getenv("NEO4J_PASSWORD"))
)

with driver.session() as session:
    # Activity IDs
    print("=== ACTIVITY NODES ===")
    result = session.run("""
        MATCH (a:Activity)
        RETURN a.id AS id, a.name AS name, a.article_ref AS article
        ORDER BY a.id
    """)
    for r in result:
        print(f"  id='{r['id']}'  name='{r['name']}'  article={r['article']}")

    # Condition IDs
    print("\n=== CONDITION NODES ===")
    result = session.run("""
        MATCH (c:Condition)
        RETURN c.id AS id, c.article_ref AS article, c.expression AS expr
        ORDER BY c.id
    """)
    for r in result:
        print(f"  id='{r['id']}'  article={r['article']}  expr={r['expr']}")

driver.close()

=== ACTIVITY NODES ===
  id='ACT-001'  name='A_Create Application'  article=Article 8(1)
  id='ACT-002'  name='A_Submitted'  article=Article 8(1)
  id='ACT-003'  name='A_Concept'  article=Article 8(1)
  id='ACT-004'  name='A_Accepted'  article=Article 8(1)
  id='ACT-005'  name='O_Create Offer'  article=Article 5(1)
  id='ACT-006'  name='O_Created'  article=Article 5(1)
  id='ACT-007'  name='O_Sent (mail and online)'  article=Article 5(1)
  id='ACT-008'  name='A_Complete'  article=Article 10(1)
  id='ACT-009'  name='O_Accepted'  article=Article 10(1)
  id='ACT-010'  name='O_Returned'  article=Article 14(1)
  id='ACT-011'  name='A_Cancelled'  article=Article 14(1)
  id='ACT-012'  name='A_Incomplete'  article=Article 7(1)
  id='ACT-013'  name='W_Call incomplete files'  article=Article 7(1)

=== CONDITION NODES ===
  id='COND-001'  article=Article 14(1)  expr=days_between(O_Sent, O_Returned) <= 14
  id='COND-002'  article=Article 8(1)  expr=A_Accepted BEFORE O_Create Offer
  id='COND-003' 

In [5]:
# Fix REQUIRES edges với đúng IDs
from neo4j import GraphDatabase
from dotenv import load_dotenv
import os

load_dotenv('../.env')
driver = GraphDatabase.driver(
    os.getenv("NEO4J_URI"),
    auth=(os.getenv("NEO4J_USERNAME"),
          os.getenv("NEO4J_PASSWORD"))
)

# Mapping đúng: Activity ID → Condition ID
REQUIRES_EDGES = [
    ('ACT-010', 'COND-001'),  # O_Returned    → 14-day withdrawal
    ('ACT-004', 'COND-002'),  # A_Accepted    → must precede O_Create Offer
    ('ACT-007', 'COND-003'),  # O_Sent        → must precede A_Complete
    ('ACT-012', 'COND-004'),  # A_Incomplete  → 8-day handling
]

with driver.session() as session:
    for act_id, cond_id in REQUIRES_EDGES:
        session.run("""
            MATCH (a:Activity {id: $act_id})
            MATCH (c:Condition {id: $cond_id})
            MERGE (a)-[:REQUIRES]->(c)
        """, act_id=act_id, cond_id=cond_id)
        print(f"  {act_id} -[:REQUIRES]-> {cond_id}")

    # Verify
    result = session.run("""
        MATCH (a:Activity)-[:REQUIRES]->(c:Condition)
        RETURN a.name AS activity, c.expression AS condition,
               c.article_ref AS article
        ORDER BY c.article_ref
    """)
    print("\n=== VERIFY REQUIRES ===")
    count = 0
    for r in result:
        print(f"  {r['activity']:<30} → {r['condition']}")
        count += 1
    print(f"\nTổng REQUIRES edges: {count}")

    # Verify tất cả edges
    print("\n=== TOÀN BỘ EDGES ===")
    for rel in ['MUST_PRECEDE', 'REQUIRES', 'PERFORMED_BY', 'DEFINED_IN']:
        r = session.run(f"MATCH ()-[r:{rel}]->() RETURN count(r) AS c").single()
        print(f"  {rel:<14}: {r['c']}")

driver.close()

  ACT-010 -[:REQUIRES]-> COND-001
  ACT-004 -[:REQUIRES]-> COND-002
  ACT-007 -[:REQUIRES]-> COND-003
  ACT-012 -[:REQUIRES]-> COND-004

=== VERIFY REQUIRES ===
  O_Returned                     → days_between(O_Sent, O_Returned) <= 14
  O_Sent (mail and online)       → O_Sent BEFORE A_Complete
  A_Incomplete                   → days_between(A_Incomplete, A_Validating) <= 8
  A_Accepted                     → A_Accepted BEFORE O_Create Offer

Tổng REQUIRES edges: 4

=== TOÀN BỘ EDGES ===
  MUST_PRECEDE  : 6
  REQUIRES      : 4
  PERFORMED_BY  : 13
  DEFINED_IN    : 17
